In [81]:
import pandas as pd
pd.set_option('display.max_rows', None)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [82]:
def clean_quantity(row):
    if pd.isna(row) or row == 'NaN':
        return None, None
    
    # Virgülü noktaya çevir (1,5 -> 1.5)
    row = str(row).replace(',', '.')
    
    # Regex ile sayısal kısmı ve birimi ayır
    # İlk sayısal grubu (ondalık dahil) ve sonrasındaki harf grubunu yakalar
    match = re.search(r"(\d+\.?\d*)\s*([a-zA-Zğüşıöç]+)?", row, re.IGNORECASE)
    
    if match:
        value = match.group(1)
        unit = match.group(2).lower() if match.group(2) else None
        
        # Birim standardizasyonu
        unit_map = {
            'ml': 'ml', 'l': 'l', 'g': 'g', 'kg': 'kg', 
            'grammes': 'g', 'gr': 'g', 'cl': 'cl'
        }
        unit = unit_map.get(unit, unit)
        
        return float(value), unit
    return None, None

def clean_categories(text):
    if pd.isna(text):
        return []
    
    # 1. Adım: Virgülleri ve boşlukları temizle
    # Virgüllere göre böl, her parçanın başındaki/sonundaki boşluğu sil
    parts = [part.strip() for part in text.split(',')]
    
    # 2. Adım: Sadece içi dolu olan (boş olmayan) kelimeleri tut
    clean_list = [p for p in parts if p and p != '']
    
    return clean_list

In [83]:
data = pd.read_csv('/Users/oguzhanerbil/Documents/Repolarım/food-health-predictor/data/ham_data.csv')

In [84]:
data.head()

,url,barkod,urun_adi,miktar,ambalaj,markalar,kategoriler,etiketler,mensei,uretim_yerleri,...,nutriscore_puan,ns_negatif_puan,ns_pozitif_puan,ns_enerji_puan,ns_seker_puan,ns_doymus_yag_puan,ns_tuz_puan,ns_protein_puan,ns_lif_puan,ns_meyve_sebze_baklagil_puan
0,https://world.openfoodfacts.org/product/611124...,6111246721261,Fromage Blanc Nature – Milky Food Professional...,1 kg,Plastic,Milky Food Professional,"Dairies, ,, Fermented foods, ,, Fermented milk...",Vegetarian,Maroc,Maroc,...,-2.0,1.0,3.0,1.0,0.0,0.0,0.0,3.0,0.0,0.0
1,https://world.openfoodfacts.org/product/611103...,6111035000430,sidi ali – سيدي علي – 33 cl,33 cl,"Plastic, ,, Bottle",سيدي علي,"Beverages and beverages preparations, ,, Bever...",NaN,NaN,NaN,...,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,https://world.openfoodfacts.org/product/611103...,6111035000058,"Eau minérale naturelle – sidi ali – 1,5 L","1,5 L","Plastic, ,, Bottle or vial, ,, Bottle",sidi ali,"Beverages and beverages preparations, ,, Bever...","ISO 22000, ,, ISO 14001, ,, ISO 45001, ,, ISO ...",Morocco,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,https://world.openfoodfacts.org/product/611103...,6111035002175,Sidi Ali – 2 L,2 L,NaN,Sidi Ali,"Beverages and beverages preparations, ,, Bever...",Green Dot,"Bassin d'Oulmès, ,, Sidi Ali Cherif",NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,https://world.openfoodfacts.org/product/327408...,3274080005003,Eau De Source – Cristaline – 1500 ml,1500 ml,"Aluminium-can, ,, HdpeFilm-packet, ,, PpFilm-w...",Cristaline,"Beverages and beverages preparations, ,, Bever...",Triman,France,"Saint-Martin de Gurson, ,, France, ,, 24610",...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [85]:
data["urun_bilgisi"] = data["urun_adi"].str.split("–").str[0].str.strip()

data[['miktar', 'birim']] = data['miktar'].apply(
    lambda x: pd.Series(clean_quantity(x))
)

data['kategori_listesi'] = data['kategoriler'].apply(clean_categories)
data['etiketler_listesi'] = data['etiketler'].apply(clean_categories)

data['alerjenler'] = data['alerjenler'].apply(clean_categories)
data['eser_miktarlar'] = data['eser_miktarlar'].apply(lambda x: [item.strip() for item in str(x).split(',')] if pd.notna(x) else [])

data['markalar'] = data['markalar'].apply(lambda x: str(x).split(',')[0].strip() if pd.notnull(x) else x)
data['markalar'] = data['markalar'].apply(lambda x: str(x).split(',')[0].strip().lower() if pd.notnull(x) else x)

In [86]:
data[data["yesil_skor_notu"].isna()].head()

,url,barkod,urun_adi,miktar,ambalaj,markalar,kategoriler,etiketler,mensei,uretim_yerleri,...,ns_seker_puan,ns_doymus_yag_puan,ns_tuz_puan,ns_protein_puan,ns_lif_puan,ns_meyve_sebze_baklagil_puan,urun_bilgisi,birim,kategori_listesi,etiketler_listesi
1,https://world.openfoodfacts.org/product/611103...,6111035000430,sidi ali – سيدي علي – 33 cl,33.0,"Plastic, ,, Bottle",سيدي علي,"Beverages and beverages preparations, ,, Bever...",NaN,NaN,NaN,...,1.0,0.0,0.0,0.0,0.0,0.0,sidi ali,cl,"[Beverages and beverages preparations, Beverag...",[]
2,https://world.openfoodfacts.org/product/611103...,6111035000058,"Eau minérale naturelle – sidi ali – 1,5 L",1.5,"Plastic, ,, Bottle or vial, ,, Bottle",sidi ali,"Beverages and beverages preparations, ,, Bever...","ISO 22000, ,, ISO 14001, ,, ISO 45001, ,, ISO ...",Morocco,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,Eau minérale naturelle,l,"[Beverages and beverages preparations, Beverag...","[ISO 22000, ISO 14001, ISO 45001, ISO 9001]"
3,https://world.openfoodfacts.org/product/611103...,6111035002175,Sidi Ali – 2 L,2.0,NaN,sidi ali,"Beverages and beverages preparations, ,, Bever...",Green Dot,"Bassin d'Oulmès, ,, Sidi Ali Cherif",NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,Sidi Ali,l,"[Beverages and beverages preparations, Beverag...",[Green Dot]
4,https://world.openfoodfacts.org/product/327408...,3274080005003,Eau De Source – Cristaline – 1500 ml,1500.0,"Aluminium-can, ,, HdpeFilm-packet, ,, PpFilm-w...",cristaline,"Beverages and beverages preparations, ,, Bever...",Triman,France,"Saint-Martin de Gurson, ,, France, ,, 24610",...,0.0,0.0,0.0,0.0,0.0,0.0,Eau De Source,ml,"[Beverages and beverages preparations, Beverag...",[Triman]
5,https://world.openfoodfacts.org/product/611112...,6111128000071,Ain Saïss – Danone – 1.5 l,1.5,Plastic bottle,danone,"Beverages and beverages preparations, ,, Bever...",NaN,Morocco,Fes Morocco,...,0.0,0.0,0.0,0.0,0.0,0.0,Ain Saïss,l,"[Beverages and beverages preparations, Beverag...",[]


In [87]:
data[["miktar","birim"]].head()

,miktar,birim
0,1.0,kg
1,33.0,cl
2,1.5,l
3,2.0,l
4,1500.0,ml


In [88]:
data.isnull().mean() * 100

url                              0.000000
barkod                           0.000000
urun_adi                         0.000000
miktar                          10.957223
ambalaj                         43.004909
markalar                         2.507013
kategoriler                      0.000000
etiketler                       19.512623
mensei                          59.256662
uretim_yerleri                  69.652875
satildigi_ulkeler                0.035063
icerik_metni                     5.855540
alerjenler                       0.000000
eser_miktarlar                   0.000000
icerik_sayisi                    6.206171
nutriscore_notu                  0.245442
nova_grubu                      11.115007
yesil_skor_notu                 26.858345
palmiye_yagi_icermez             9.344320
vejetaryen                      13.990182
vegan_durumu                     6.030856
yag_seviyesi                     4.242637
doymus_yag_seviyesi              4.996494
seker_seviyesi                   4

In [89]:
data.drop(columns=["sodyum_g","enerji_kj","yag_seviyesi","doymus_yag_seviyesi","seker_seviyesi","tuz_seviyesi","url","urun_adi","kategoriler","etiketler","ambalaj","mensei","uretim_yerleri","satildigi_ulkeler","icerik_metni","urun_bilgisi","nutriscore_puan","ns_negatif_puan","ns_pozitif_puan","ns_enerji_puan","ns_seker_puan","ns_doymus_yag_puan","ns_tuz_puan","ns_protein_puan","ns_lif_puan","ns_meyve_sebze_baklagil_puan"], inplace=True)

In [90]:
data["etiketler_listesi"].head(20)

0                                          [Vegetarian]
1                                                    []
2           [ISO 22000, ISO 14001, ISO 45001, ISO 9001]
3                                           [Green Dot]
4                                              [Triman]
5                                                    []
6                                                    []
7                                                    []
8                                    [Green Dot, Maroc]
9                                                    []
10                                          [Green Dot]
11    [French milk, Made in France, Nutriscore, Nutr...
12    [Fair trade, Source of fibre, High fibres, Mad...
13    [Vegetarian, Fair trade, No gluten, Organic, C...
14    [ISO 22000, ISO 14001, ISO 45001, ISO 9001, Na...
15    [Sustainable, No preservatives, Source of fibr...
16    [No gluten, No preservatives, FSC, Green Dot, ...
17                                              

In [91]:
for column in data.columns:
    unique_values = data[column].astype(str).unique()
    unique_values_str = sorted(list(unique_values))
    n = 10
    displayed_values = unique_values_str[:n]
    print(f"{column} ({len(unique_values_str)}): {displayed_values}{' ...' if len(unique_values_str) > n else ''}")

barkod (5704): ['10001219', '10001400', '10015599', '10089149', '10096376', '10100295', '10217603', '10254899', '10677964', '11110016508'] ...
miktar (321): ['0.0', '0.07', '0.095', '0.1', '0.112', '0.125', '0.14', '0.15', '0.18', '0.2'] ...
markalar (1762): ['07x netto 03.25', '10dh', '166', '2keep', '365 whole foods market', '3bears', '5 fruits', '7 up', 'abatilles', "abbot kinney's"] ...
alerjenler (305): ["['Apple', 'Gluten', 'Gluten', 'Gluten']", "['Apple', 'Milk', 'Milk']", "['Apple']", "['Avoine']", "['Banana']", "['Beef', 'Gluten', 'Milk', 'Pork', 'Sulphur dioxide and sulphites']", "['Beef', 'Gluten', 'Sesame seeds', 'Soybeans']", "['Beef', 'Milk', 'Pork']", "['Beef']", "['Celery', 'Chicken', 'Eggs', 'Gluten', 'Milk', 'Soybeans']"] ...
eser_miktarlar (478): ["['Agua']", "['Basil-oil', 'Pesticides']", "['Celery', 'Crustaceans', 'Eggs', 'Fish', 'Gluten', 'Lupin', 'Milk', 'Molluscs', 'Mustard', 'Nuts', 'Peanuts', 'Sesame seeds', 'Soybeans', 'Sulphur dioxide and sulphites']", "['Ce

In [92]:
data["palmiye_yagi_icermez"].value_counts()

palmiye_yagi_icermez
True     4385
False     786
Name: count, dtype: int64

In [93]:
data.isnull().mean() * 100

barkod                         0.000000
miktar                        10.957223
markalar                       2.507013
alerjenler                     0.000000
eser_miktarlar                 0.000000
icerik_sayisi                  6.206171
nutriscore_notu                0.245442
nova_grubu                    11.115007
yesil_skor_notu               26.858345
palmiye_yagi_icermez           9.344320
vejetaryen                    13.990182
vegan_durumu                   6.030856
enerji_kcal                    3.190743
yag_g                          3.208275
doymus_yag_g                   3.997195
karbonhidrat_g                 3.278401
seker_g                        3.962132
lif_g                         19.950912
protein_g                      3.208275
tuz_g                          1.840813
alkol_yuzde                   93.267882
meyve_sebze_baklagil_yuzde    73.825386
birim                         12.903226
kategori_listesi               0.000000
etiketler_listesi              0.000000


In [94]:
data.to_csv('/Users/oguzhanerbil/Documents/Repolarım/food-health-predictor/data/data.csv', index=False)

In [95]:
data.columns

Index(['barkod', 'miktar', 'markalar', 'alerjenler', 'eser_miktarlar',
       'icerik_sayisi', 'nutriscore_notu', 'nova_grubu', 'yesil_skor_notu',
       'palmiye_yagi_icermez', 'vejetaryen', 'vegan_durumu', 'enerji_kcal',
       'yag_g', 'doymus_yag_g', 'karbonhidrat_g', 'seker_g', 'lif_g',
       'protein_g', 'tuz_g', 'alkol_yuzde', 'meyve_sebze_baklagil_yuzde',
       'birim', 'kategori_listesi', 'etiketler_listesi'],
      dtype='object')